In [1]:
# %% 1. Setup: works in both Colab and VS Code
import os
import sys
import ast
import glob
import zipfile

import pandas as pd
from IPython.display import Audio, display

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP_DIR = "/content/drive/MyDrive/dl-final/data"   # where the zips live on Drive
    WORK_DIR = "/content/raw"                          # fast temporary disk
else:
    # Local: resolve the repo's data/ folder no matter where the notebook lives
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    ZIP_DIR = os.path.join(REPO_ROOT, "data")
    WORK_DIR = os.path.join(REPO_ROOT, "data")
os.makedirs(WORK_DIR, exist_ok=True)
METADATA_ZIP = os.path.join(ZIP_DIR, "fma_metadata.zip")
AUDIO_ZIP = os.path.join(ZIP_DIR, "fma_medium.zip")
TRACKS_CSV = os.path.join(WORK_DIR, "fma_metadata", "tracks.csv")
GENRES_CSV = os.path.join(WORK_DIR, "fma_metadata", "genres.csv")
AUDIO_DIR = os.path.join(WORK_DIR, "fma_medium")


# %% 2. Extract only the files we need (skips anything already extracted)
def extract_members(zip_path, prefix, dest):
    """Extract only the files inside zip_path whose names start with prefix."""
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        members = [m for m in names if m.startswith(prefix)]
        if not members:
            print(f"Nothing matched '{prefix}'. First entries in the zip:", names[:5])
            return
        new = [m for m in members if not os.path.exists(os.path.join(dest, m))]
        for m in new:
            zf.extract(m, dest)
    print(f"{prefix}: {len(members)} files ({len(new)} newly extracted)")


extract_members(METADATA_ZIP, "fma_metadata/tracks.csv", WORK_DIR)
extract_members(METADATA_ZIP, "fma_metadata/genres.csv", WORK_DIR)

if os.path.exists(AUDIO_ZIP):
    extract_members(AUDIO_ZIP, "fma_medium/000/", WORK_DIR)  # one folder, roughly 160 clips
else:
    print("No audio zip here. To listen locally, put a few MP3s in", os.path.join(AUDIO_DIR, "000"))


# %% 3. Load the labels
tracks = pd.read_csv(TRACKS_CSV, index_col=0, header=[0, 1])
genres = pd.read_csv(GENRES_CSV, index_col=0)

# The medium set includes the small set's tracks too
medium = tracks[tracks[("set", "subset")].isin(["small", "medium"])]
print(len(medium), "tracks in the medium subset\n")
print(medium[("track", "genre_top")].value_counts())


# %% 4. Helpers
def track_id_from_path(path):
    """000002.mp3 -> 2"""
    return int(os.path.splitext(os.path.basename(path))[0])


def info(track_id):
    row = tracks.loc[track_id]
    all_ids = ast.literal_eval(row[("track", "genres_all")])  # stored as text like "[21, 811]"
    return {
        "track_id": track_id,
        "title": row[("track", "title")],
        "artist": row[("artist", "name")],
        "genre_top": row[("track", "genre_top")],
        "all_genres": [genres.loc[g, "title"] for g in all_ids],
        "split": row[("set", "split")],
    }


def play(path):
    d = info(track_id_from_path(path))
    print(f"[{d['track_id']}] {d['title']}, by {d['artist']}")
    print(f"  genre_top: {d['genre_top']}  |  all genres: {', '.join(d['all_genres'])}  |  split: {d['split']}")
    display(Audio(path))


def browse(n=5, genre=None, folder="000"):
    paths = sorted(glob.glob(os.path.join(AUDIO_DIR, folder, "*.mp3")))
    if genre is not None:
        paths = [p for p in paths if info(track_id_from_path(p))["genre_top"] == genre]
    print(f"{len(paths)} clips found; showing {min(n, len(paths))}\n")
    for p in paths[:n]:
        play(p)


# %% 5. Listen
browse(n=5)
# browse(n=5, genre="Hip-Hop")
# play(os.path.join(AUDIO_DIR, "000", "000002.mp3"))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/msmith05/Documents/03 Projects/tiny-timbre/data/fma_metadata.zip'